# FLAG 2027 — NB-1: re-extract features from the raw media

Plan: `docs/PLAN_KAGGLE.md`. Runs **once**; its output folder becomes the Kaggle dataset `flag2027-feats-v2`
that NB-2 / NB-3 read (so no training notebook ever touches wav/jpg again).

**Settings:** Accelerator **GPU T4 x2** (one GPU is used), **Internet ON** (pretrained encoders),
**Persistence: Files** is not needed. Input: the raw dataset (`Input/` = `train_set.zip` + `dev_set.zip` +
`meta_file_train_set.csv`; Kaggle may auto-extract, both forms work).

| output | shape | what |
|---|---|---|
| `voice_ecapa192_<split>.npy` | N x 192 | ECAPA output — cross-check vs organiser CSV |
| `voice_ecapa6144_<split>.npy` | N x 6144 | ECAPA attentive-stat-pooling (FAME 2026 winner's layer) |
| `voice_ecapa{192,6144}crop_train.npy` | 3 x N x D | train utterances cropped to the dev-**Bangla** duration distribution |
| `face_arcface_<split>.npy` | N x 512 | ArcFace R100, SCRFD detection + 5-point alignment |
| `voice_<ssl>_<split>.npy` | N x (L+1) x H fp16 | per-layer mean of WavLM-Large / XLS-R-300m (multilingual) |

**OOM safety:** audio sorted by length and packed by *padded seconds* (the longest dev file is 81.6 s);
CUDA OOM halves the batch; a single file that still fails is run on CPU. SSL models use 20 s windows.
**Quality checks printed:** batched-vs-single cosine (must be > 0.999), face detection rate, fp16-vs-fp32 cosine.
**Resumable:** every array is written as soon as it is done and skipped on re-run.

Expected time on a T4: ECAPA + crops ~15 min, ArcFace ~10 min, each SSL model ~15-25 min.

In [1]:
import os, sys
from pathlib import Path
LOCAL = os.environ.get("FLAG_LOCAL") == "1"          # CPU smoke test on a mini copy of the data
IN = Path(os.environ.get("FLAG_IN", "/kaggle/input"))
WORKDIR = Path(os.environ.get("FLAG_OUT", "/kaggle/working"))
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
print("LOCAL" if LOCAL else "KAGGLE", "| input", IN, "| working", WORKDIR)

KAGGLE | input /kaggle/input | working /kaggle/working


In [2]:
# Writes the helper modules next to this notebook (no .py upload needed).
import base64
open('flag_lib.py', 'wb').write(base64.b64decode(''.join('''
...
cmV0dXJuIHBhdGgK
'''.split())))
open('flag_extract.py', 'wb').write(base64.b64decode(''.join('''
...
'''.split())))
print('written:', ['flag_lib.py', 'flag_extract.py'])

written: ['flag_lib.py', 'flag_extract.py']


## 1. Setup and index (row order = organiser txt)

In [3]:
# -U matters: Kaggle preinstalls an old speechbrain (<1.1.1 breaks on torchaudio 2.9+).
# insightface is pinned and installed --no-deps: its declared deps would pull CPU `onnxruntime` over
# `onnxruntime-gpu` and a second OpenCV; Kaggle already ships scipy / scikit-image / opencv.
if not LOCAL:
    !pip -q install -U "speechbrain>=1.1.1" onnxruntime-gpu onnx 2>&1 | tail -3
    !pip -q install --no-deps "insightface==2.0" 2>&1 | tail -3
import numpy as np, torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

google-cloud-aiplatform 1.148.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 7.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.1 MB/s eta 0:00:00
torch 2.10.0+cu128 | cuda True Tesla T4


In [4]:
import flag_extract as X
OUT = WORKDIR / "feats"; OUT.mkdir(exist_ok=True)
N_CROPS = 3                      # duration-matched crops per train utterance
# multilingual SSL encoders (plan H2b). Set to [] to skip. Local smoke test uses a small model.
SSL = [("microsoft/wavlm-large", "wavlm"), ("facebook/wav2vec2-xls-r-300m", "xlsr")]
if LOCAL:
    SSL = [("microsoft/wavlm-base-plus", "wavlmbp")]
MAX_PADDED_SEC = 240.0           # per ECAPA batch; OOM halves it automatically
root = X.media_root(IN, WORKDIR / "raw")
S = X.index_splits(root)
X.copy_metadata(IN, S, OUT)
print("total files per modality:", sum(len(v["wav"]) for v in S.values()))

11:56:44 train                 6485 rows | dur median 5.2s max 60.0s
11:56:50 no_gender/English     1008 rows | dur median 5.8s max 81.6s
11:57:00 no_gender/Bangla      1406 rows | dur median 4.7s max 43.9s
11:57:07 gender/English         982 rows | dur median 6.1s max 81.6s
11:57:15 gender/Bangla         1468 rows | dur median 4.7s max 43.9s
total files per modality: 11349


## 2. ECAPA-TDNN: 192-d + 6144-d, plus duration-matched train crops

In [5]:
ecapa = X.extract_ecapa(S, OUT, n_crops=N_CROPS, max_padded_sec=MAX_PADDED_SEC)
CONS = X.check_batch_consistency(ecapa, S, n=12 if LOCAL else 24)
# local CPU test measured 0.99969 / 0.99989 (fp32 reduction order differs between batch sizes)
assert min(CONS.values()) > 0.995, f"padding changes the embedding: {CONS} - do not use these features"
if min(CONS.values()) < 0.999:
    print("WARNING: batched vs single cosine below 0.999:", CONS)
del ecapa; torch.cuda.empty_cache()

hyperparams.yaml: 0.00B [00:00, ?B/s]

embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

mean_var_norm_emb.ckpt:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

classifier.ckpt:   0%|          | 0.00/5.53M [00:00<?, ?B/s]

label_encoder.txt: 0.00B [00:00, ?B/s]

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


11:57:29   ecapa train 4/6485  2s  eta 3027s  oom=0
11:57:54   ecapa train 445/6485  26s  eta 357s  oom=0
11:58:19   ecapa train 1387/6485  52s  eta 190s  oom=0
11:58:45   ecapa train 2892/6485  78s  eta 97s  oom=0
11:59:06   ecapa train 4492/6485  98s  eta 43s  oom=0
11:59:19   ecapa train 6092/6485  112s  eta 7s  oom=0
11:59:22   ecapa train 6485/6485  114s  eta 0s  oom=0
11:59:22 train saved (6485, 6144)
11:59:22   ecapa no_gender/English 2/1008  0s  eta 182s  oom=0
11:59:42   ecapa no_gender/English 1008/1008  19s  eta 0s  oom=0
11:59:42 no_gender/English saved (1008, 6144)
11:59:42   ecapa no_gender/Bangla 5/1406  0s  eta 121s  oom=0
12:00:02   ecapa no_gender/Bangla 1406/1406  21s  eta 0s  oom=0
12:00:02 no_gender/Bangla saved (1406, 6144)
12:00:03   ecapa gender/English 2/982  0s  eta 175s  oom=0
12:00:23   ecapa gender/English 982/982  20s  eta 0s  oom=0
12:00:23 gender/English saved (982, 6144)
12:00:23   ecapa gender/Bangla 5/1468  0s  eta 132s  oom=0
12:00:45   ecapa gender/

In [6]:
# Is the organisers' 192-d CSV this same ECAPA?  ~1.0 -> yes, and 6144-d really is "the layer before".
import pandas as pd
given = pd.read_csv(sorted(IN.rglob("train_English_voices.csv"))[0], header=None).iloc[:, :-1].values
ours = np.load(OUT / "voice_ecapa192_train.npy")
assert len(given) == len(ours)
unit = lambda A: A / np.linalg.norm(A, axis=1, keepdims=True)
cos = np.sum(unit(given) * unit(ours), 1)
ENC_MATCH = dict(mean=float(cos.mean()), p5=float(np.percentile(cos, 5)))
print("cos(ours ECAPA-192, organiser CSV):", ENC_MATCH)
# Local mini test gave mean ~0.02 while both give the same speaker EER (5.5%): rows ARE aligned, the
# organisers simply used a different ECAPA checkpoint. NB-2 P0 re-checks alignment via unimodal EER.
if ENC_MATCH["mean"] < 0.9:
    print("-> different encoder from speechbrain/spkrec-ecapa-voxceleb (expected); alignment is checked in NB-2 P0")

cos(ours ECAPA-192, organiser CSV): {'mean': 0.018285131243549125, 'p5': -0.09994318853130563}
-> different encoder from speechbrain/spkrec-ecapa-voxceleb (expected); alignment is checked in NB-2 P0


## 3. ArcFace R100 with detection + alignment

In [7]:
DET = X.extract_arcface(S, OUT)
for k, v in DET.items():
    if v < 0.9:
        print(f"WARNING {k}: only {100*v:.1f}% faces detected+aligned; the rest used a resized centre crop")

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:02<00:00, 114269.50KB/s]


Failed to load libcublasLt.so.13: libcublasLt.so.13: cannot open shared object file: No such file or directory
Failed to load libcublas.so.13: libcublas.so.13: cannot open shared object file: No such file or directory
Failed to load libnvrtc.so.13: libnvrtc.so.13: cannot open shared object file: No such file or directory
Failed to load libcufft.so.12: libcufft.so.12: cannot open shared object file: No such file or directory
Failed to load libcudart.so.13: libcudart.so.13: cannot open shared object file: No such file or directory
Please follow https://onnxruntime.ai/docs/install/#cuda-and-cudnn to install CUDA.


2026-09-25 12:04:01.848474412 [E:onnxruntime:Default, provider_bridge_ort.cc:2395 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1988 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library /usr/local/lib/python3.12/dist-packages/onnxruntime/capi/libonnxruntime_providers_cuda.so with error: libcublasLt.so.13: cannot open shared object file: No such file or directory

2026-09-25 12:04:02.341463872 [E:onnxruntime:Default, provider_bridge_ort.cc:2395 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1988 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library /usr/local/lib/python3.12/dist-packages/onnxruntime/capi/libonnxruntime_providers_cuda.so with error: libcublasLt.so.13: cannot open shared object file: No such file or directory

2026-09-25 12:04:02.383910217 [E:onnxruntime:Default, prov

12:04:03 onnxruntime providers available ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider'] | recognition runs on ['CPUExecutionProvider']


2026-09-25 12:04:02.605799243 [E:onnxruntime:Default, provider_bridge_ort.cc:2395 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1988 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library /usr/local/lib/python3.12/dist-packages/onnxruntime/capi/libonnxruntime_providers_cuda.so with error: libcublasLt.so.13: cannot open shared object file: No such file or directory



12:04:05   arcface train 0/6485 2s
12:07:54   arcface train 1000/6485 232s
12:11:36   arcface train 2000/6485 454s
12:15:21   arcface train 3000/6485 678s
12:19:09   arcface train 4000/6485 906s
12:22:59   arcface train 5000/6485 1136s
12:26:43   arcface train 6000/6485 1360s
12:28:32 train: detection+alignment rate 100.0%
12:28:32   arcface no_gender/English 0/1008 0s
12:32:19   arcface no_gender/English 1000/1008 228s
12:32:21 no_gender/English: detection+alignment rate 100.0%
12:32:21   arcface no_gender/Bangla 0/1406 0s
12:36:06   arcface no_gender/Bangla 1000/1406 225s
12:37:38 no_gender/Bangla: detection+alignment rate 100.0%
12:37:38   arcface gender/English 0/982 0s
12:41:17 gender/English: detection+alignment rate 99.8%
12:41:17   arcface gender/Bangla 0/1468 0s
12:45:04   arcface gender/Bangla 1000/1468 227s
12:46:50 gender/Bangla: detection+alignment rate 100.0%


## 4. Multilingual self-supervised encoders (optional, H2b)

In [8]:
SSL_COS = {}
for model_id, short in SSL:
    try:
        SSL_COS[short] = X.extract_ssl(S, OUT, model_id, short)
    except Exception as e:                       # an optional encoder must not kill the run
        print(f"SSL {model_id} failed: {type(e).__name__}: {e}")

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

...

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


13:03:14 xlsr: fp16 vs fp32 per-layer cosine min 0.99996
13:03:14   xlsr train 0/6485 0s
13:04:23   xlsr train 1000/6485 69s
13:05:31   xlsr train 2000/6485 137s
13:06:39   xlsr train 3000/6485 205s
13:07:46   xlsr train 4000/6485 272s
13:08:56   xlsr train 5000/6485 342s
13:10:03   xlsr train 6000/6485 409s
13:10:38 train saved (6485, 25, 1024)
13:10:38   xlsr no_gender/English 0/1008 0s
13:11:51   xlsr no_gender/English 1000/1008 73s
13:11:51 no_gender/English saved (1008, 25, 1024)
13:11:51   xlsr no_gender/Bangla 0/1406 0s
13:12:49   xlsr no_gender/Bangla 1000/1406 57s
13:13:12 no_gender/Bangla saved (1406, 25, 1024)
13:13:12   xlsr gender/English 0/982 0s
13:14:26 gender/English saved (982, 25, 1024)
13:14:27   xlsr gender/Bangla 0/1468 0s
13:15:24   xlsr gender/Bangla 1000/1468 58s
13:15:51 gender/Bangla saved (1468, 25, 1024)


## 5. Manifest + clean-up

In [9]:
import shutil
man = X.write_manifest(OUT, dict(consistency=CONS, encoder_match=ENC_MATCH, face_detect=DET, ssl_fp16_cos=SSL_COS,
                                 n_crops=N_CROPS))
# keep the output small: drop extracted raw media and model checkpoints from /kaggle/working
for p in [WORKDIR / "raw", WORKDIR / "_ecapa_ckpt", OUT.parent / "_ecapa_ckpt"]:
    shutil.rmtree(p, ignore_errors=True)
tot = sum(p.stat().st_size for p in OUT.glob("*")) / 1e9
print(f"\n{len(man['files'])} arrays, {tot:.2f} GB in {OUT}")
print("NEXT: Save Version (Save & Run All) -> open the version -> Output -> 'New Dataset' -> name it flag2027-feats-v2")

13:15:52 manifest: 33 arrays

33 arrays, 1.97 GB in /kaggle/working/feats
NEXT: Save Version (Save & Run All) -> open the version -> Output -> 'New Dataset' -> name it flag2027-feats-v2
